# 3D Breast Surface Reconstruction (Corrected)
This notebook implements the methodology of Costa et al. (2023) to reconstruct a 3D breast surface from 5 thermographic views.
It includes critical bug fixes for issues present in the paper's original description:
1. **P1/P3 assignment**: Correctly anchors right views to the right breast and left views to the left breast.
2. **Lateral edge extraction**: Extracts the front profile of the breast instead of just the bottom fold.
3. **Rotation directions**: Adjusts rotation signs so the breasts protrude forward (Z <= 0) without crossing over.



In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import splprep, splev
import os

# Define the image paths (Absolute paths)
BASE_DIR = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction"
patient_views = {
    0.0: os.path.join(BASE_DIR, r"data\Automated_Edges3\Patient_13\benign\Anterior (Front)_edge.png"),
    45.0: os.path.join(BASE_DIR, r"data\Automated_Edges3\Patient_13\benign\Right Oblique (45°)_edge.png"),
    90.0: os.path.join(BASE_DIR, r"data\Automated_Edges3\Patient_13\benign\Right Lateral (90°)_edge.png"),
    -45.0: os.path.join(BASE_DIR, r"data\Automated_Edges3\Patient_13\benign\Left Oblique (45°)_edge.png"),
    -90.0: os.path.join(BASE_DIR, r"data\Automated_Edges3\Patient_13\benign\Left Lateral (90°)_edge.png"),
}

for deg, path in patient_views.items():
    if not os.path.exists(path):
        print(f"WARNING: File not found: {path}")


### Phase 1: Edge Extraction & Ordering


In [8]:
def _order_curve_8_connected(points):
    """Order unordered edge pixels into a near-continuous 8-connected path."""
    if len(points) == 0:
        return np.empty((0, 2), dtype=int)
    
    pts = [tuple(map(int, p)) for p in points]
    pts = list(dict.fromkeys(pts))
    point_set = set(pts)
    
    neighbor_offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    ]
    
    neighbors = {p: [] for p in pts}
    for p in pts:
        x, y = p
        for dx, dy in neighbor_offsets:
            q = (x + dx, y + dy)
            if q in point_set:
                neighbors[p].append(q)
                
    endpoints = [p for p in pts if len(neighbors[p]) <= 1]
    if endpoints:
        start = min(endpoints, key=lambda t: t[0])
    else:
        start = min(pts, key=lambda t: t[0])
        
    ordered = [start]
    visited = {start}
    current = start
    
    while len(visited) < len(point_set):
        candidates = [q for q in neighbors[current] if q not in visited]
        if not candidates:
            unvisited = [q for q in pts if q not in visited]
            if not unvisited: break
            current = min(unvisited, key=lambda q: (q[0]-current[0])**2 + (q[1]-current[1])**2)
            ordered.append(current)
            visited.add(current)
            continue
            
        next_p = min(candidates, key=lambda q: (q[0]-current[0])**2 + (q[1]-current[1])**2)
        ordered.append(next_p)
        visited.add(next_p)
        current = next_p
        
    return np.array(ordered, dtype=int)

def extract_relevant_edge(edge_img, view_deg):
    """
    Extract the relevant fold/profile from the edge image.
    - 0, 45, -45: Bottom fold (max Y for each X)
    - 90: Right profile (max X for each Y)
    - -90: Left profile (min X for each Y)
    """
    h, w = edge_img.shape
    pts = set()
    
    if view_deg in (0.0, 45.0, -45.0):
        for x in range(w):
            y_idx = np.where(edge_img[:, x] > 127)[0]
            if len(y_idx) > 0:
                pts.add((x, int(np.max(y_idx))))
    elif view_deg == 90.0:
        for y in range(h):
            x_idx = np.where(edge_img[y, :] > 127)[0]
            if len(x_idx) > 0:
                pts.add((int(np.max(x_idx)), y))
    elif view_deg == -90.0:
        for y in range(h):
            x_idx = np.where(edge_img[y, :] > 127)[0]
            if len(x_idx) > 0:
                pts.add((int(np.min(x_idx)), y))
                
    if not pts:
        return np.empty((0, 2), dtype=int)
        
    ordered = _order_curve_8_connected(np.array(list(pts), dtype=int))
    return ordered

def get_keypoints(curve, view_deg):
    """
    Extract local anchors P1, P2, P3 from the curve.
    Returns (P_left, P_center, P_right).
    """
    if len(curve) == 0:
        return None, None, None
        
    if view_deg in (0.0, 45.0, -45.0):
        p_center = curve[np.argmin(curve[:, 1])] # min Y (highest point)
        p_left = curve[np.argmin(curve[:, 0])]   # min X (patient's right breast)
        p_right = curve[np.argmax(curve[:, 0])]  # max X (patient's left breast)
        return p_left, p_center, p_right
    elif view_deg in (90.0, -90.0):
        p_center = curve[np.argmax(curve[:, 1])] # max Y (lowest point)
        p_left = curve[0]
        p_right = curve[-1]
        return p_left, p_center, p_right
    return None, None, None



### Phase 2: 3D Geometric Transformation


In [9]:
def generate_9_curves_3d(views):
    curves = []
    
    # --- Frontal (0) ---
    img_0 = cv2.imread(views[0.0], 0)
    curve_0 = extract_relevant_edge(img_0, 0.0)
    p_left_0, p_center_0, p_right_0 = get_keypoints(curve_0, 0.0)
    
    # C1: Right Breast Base (P_left to P_center)
    idx_l = np.argmin(np.linalg.norm(curve_0 - p_left_0, axis=1))
    idx_c = np.argmin(np.linalg.norm(curve_0 - p_center_0, axis=1))
    idx_r = np.argmin(np.linalg.norm(curve_0 - p_right_0, axis=1))
    
    c1 = curve_0[min(idx_l, idx_c) : max(idx_l, idx_c)+1]
    curves.append(np.column_stack((c1, np.zeros(len(c1)))))
    
    # C2: Left Breast Base (P_center to P_right)
    c2 = curve_0[min(idx_c, idx_r) : max(idx_c, idx_r)+1]
    curves.append(np.column_stack((c2, np.zeros(len(c2)))))
    
    # C3: Midline
    top_y = float(curve_0[:, 1].min()) - 50 # Extend slightly up
    mid_y = np.linspace(p_center_0[1], top_y, 50)
    curves.append(np.column_stack((np.full(50, p_center_0[0]), mid_y, np.zeros(50))))
    
    # --- Rotated Curves ---
    # (source_angle, anchor_point, rotation_angle)
    # Note: Right breast anchor = p_left_0. Left breast anchor = p_right_0.
    rules = [
        ( 45.0, p_left_0,   45.0, "C4: Right Oblique"),
        ( 90.0, p_left_0,  -90.0, "C5: Right Lateral"),
        (-45.0, p_right_0, -45.0, "C6: Left Oblique"),
        (-90.0, p_right_0,  90.0, "C7: Left Lateral"),
        ( 45.0, p_left_0,  135.0, "C8: Aux Right"),
        (-45.0, p_right_0,-135.0, "C9: Aux Left"),
    ]
    
    for src_ang, anchor, rot, name in rules:
        img_src = cv2.imread(views[src_ang], 0)
        active = extract_relevant_edge(img_src, src_ang)
        if len(active) == 0: continue
            
        _, p_center_s, _ = get_keypoints(active, src_ang)
        
        # Center at local P2
        x_c = active[:, 0] - p_center_s[0]
        y_c = active[:, 1] - p_center_s[1]
        
        # Rotate
        rad = np.radians(rot)
        x_rot = x_c * np.cos(rad)
        z_rot = x_c * np.sin(rad) # Positive sin keeps Z negative
        
        # Translate to global anchor
        x_f = x_rot + anchor[0]
        y_f = y_c + anchor[1]
        z_f = z_rot
        
        # Filter Z <= 0 (protruding forward)
        mask = z_f <= 0
        if mask.sum() > 0:
            curves.append(np.column_stack((x_f[mask], y_f[mask], z_f[mask])))
            
    return curves

curves_3d = generate_9_curves_3d(patient_views)
print(f"Successfully generated {len(curves_3d)} 3D curves.")



Successfully generated 9 3D curves.


### Phase 3: B-Spline Smoothing & Visualization


In [10]:
def fit_bspline(pts_3d, degree=3, n_eval=100):
    if len(pts_3d) < degree + 1:
        return pts_3d
    
    # Subsample to smooth out noise
    if len(pts_3d) > 100:
        idx = np.linspace(0, len(pts_3d)-1, 100, dtype=int)
        pts_3d = pts_3d[idx]
        
    tck, u = splprep([pts_3d[:,0], pts_3d[:,1], pts_3d[:,2]], s=len(pts_3d)*2)
    u_new = np.linspace(0, 1, n_eval)
    x, y, z = splev(u_new, tck)
    return np.column_stack((x, y, z))

smooth_curves = [fit_bspline(c) for c in curves_3d]

fig = go.Figure()
colors = ['blue', 'blue', 'cyan', 'yellow', 'orange', 'green', 'red', 'purple', 'magenta']
names = ['Right Base', 'Left Base', 'Midline', 'Right Oblique', 'Right Lateral', 'Left Oblique', 'Left Lateral', 'Aux Right', 'Aux Left']

for i, c in enumerate(smooth_curves):
    name = names[i] if i < len(names) else f"Curve {i+1}"
    color = colors[i] if i < len(colors) else 'black'
    fig.add_trace(go.Scatter3d(
        x=c[:, 0], y=c[:, 1], z=c[:, 2],
        mode='lines',
        line=dict(width=5, color=color),
        name=name
    ))

# Plot anchors
img_0 = cv2.imread(patient_views[0.0], 0)
c0 = extract_relevant_edge(img_0, 0.0)
p_left, p_center, p_right = get_keypoints(c0, 0.0)

fig.add_trace(go.Scatter3d(
    x=[p_left[0], p_center[0], p_right[0]],
    y=[p_left[1], p_center[1], p_right[1]],
    z=[0, 0, 0],
    mode='markers+text',
    marker=dict(size=8, color=['red', 'yellow', 'green'], symbol='diamond'),
    text=['Right Anchor', 'Center', 'Left Anchor'],
    textposition='top center',
    name='Anchors'
))

# Note: plotly Y axis points UP by default, so we reverse it to match image coords
fig.update_layout(
    scene=dict(
        xaxis_title='X (horizontal)',
        yaxis_title='Y (vertical / down)',
        zaxis_title='Z (depth)',
        yaxis=dict(autorange='reversed')
    ),
    title="Corrected 3D Breast Curves (Costa et al. 2023)",
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()

